# Model selection: immunogenicity tools: Nanobodies
This script aims to perform model selection on immunogenicity prediction tools. Previously different tools and settings have been tested. Comparision to anti-drug antibody (ADA) data using mainly spearman rank correlation, but also Mean Absolute Rank Error (MARE), to the tool scores has resulted in the tools and settings selected for testing in this script. Here the intention is to combine the tools to one predictive model, and perform ridge regression to investigate which tools have the most infulence on the best predicted response. Here ADA data will again be used as the correct/observed response varaible. In difference to previous testing the raw ADA score as well as the raw tool output score will be used in the model selection.\
\
The selected tools and settings for this secondary testing are\
1. netMHC1_EL_pep9:\
netMHCpan for MHC class 1, prediction type EL (elution ligand), peptide length 9, sliding window 5 and the only aviable pre prepare human allel panel with 27 alleles. These are the default settings for this tool.For this prediction 3 types of score are provided: the precentile_EL score, the immunogenicity score and the MHC pre processing score.\
2. netMHC1_pep14:\
This the same settings as scores above, expcet that the peptide length is 14. This are the best scoring settings for this tool\
3. netMHC_II_EL_pep12:\
netMHCpan for MHC class 2, prediciton type EL, peptide length 12, sliding window 5 and human 27 allele panel. For this tool there are two huaman allele panels, either 7 or 27. the human 27 allele panel scored marginally higher with all other combination of settings. Further, for this tool there are 2 different scores: the precentile_EL score and the immunogenicity score. THe MHC pre processing score is not availabe for these settings, only for peptide lengths 13 to 23. This tool and setting combination was the absolute highest scoring one in the initail testing\
4. netMHC_II_EL_pep15:\
This is the same settings as above (3.) but with peptide length 15. Here there are 3 scores: preceintile_EL, immunogenicity and MHC pre processing. This are the default settings for this tool.\
5. biophi_KabKabRelaxed:\
BioPhi OASis predicts how human a peptide is. The numbering scheme is Kabat, CDR definition Kabat and the human percent threshold is set to relaxed (>50%). This is the default setttings for this tool. This tool did not score very high on the spearman rank correlation in the initial testing, however that was calcualted on antibodies only and the main concern here is nanobodies. This tool had the best MARE scores for nanobodies and is therefore included here.\
6. biophi_KabKabStrict\
BioPhi OASis predicts how human a peptide is. The numbering scheme is Kabat, CDR definition Kabat and the human percent threshold is set to strict (>10%). This is the highest scoring settings for this tool.\
7. waltz\
This is a aggregation prediciton tool that has been included because it might improve the immunogenicity predicitons. Default settings are used.\

In [1]:
# load libaries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [3]:
netMHC1_EL_pep9 = pd.read_csv("tool_outputs/NB_netMHC_peplen9_18_03_2026.csv")
netMHC1_EL_pep14 = pd.read_csv("tool_outputs/NB_netMHC_peplen14_18_03_2026.csv")
netMHC_II_EL_pep12 = pd.read_csv("tool_outputs/NB_netMHC_II_peplen12_18_03_2026.csv")
netMHC_II_EL_pep15 = pd.read_csv("tool_outputs/NB_netMHC_II_peplen15_18_03_2026.csv")
biophi_KabKabRelaxed = pd.read_excel("tool_outputs/NB_NrKab_CDRKab_relaxed_18_03_2026.xlsx")
biophi_KabKabStrict = pd.read_excel("tool_outputs/NB_NrKab_CDRKab_strict_18_03_2026.xlsx")
waltz = pd.read_csv("tool_outputs/NB_waltz.txt", sep="\t", header=None, names=["antibody", "waltz_score", "..."])

# load table with the antibody names
seqTable = pd.read_csv("tool_outputs/NB_seqTable_netMHC_II_peplen12.csv")

# load file containing the antibody name and their observed ADA percantage
ADA = pd.read_csv("../../../data/ADA_6NB.txt", sep="\t", header=None, names=["antibody", "ADA_percentage"])

In [4]:
# Sanity checks

# Check nr of antibodies
if netMHC1_EL_pep14['seq #'].nunique()!=6:
        print( "netMHC1_EL_pep14 does not have 6 antibodies")
if netMHC1_EL_pep9['seq #'].nunique()!=6:
        print( "netMHC1_EL_pep9 does not have 6 antibodies")
if netMHC_II_EL_pep12['seq #'].nunique()!=6:
        print( "netMHC_II_EL_pep12 does not have 6 antibodies")   
if netMHC_II_EL_pep15['seq #'].nunique()!=6:
        print( "netMHC_II_EL_pep15 does not have 6 antibodies")
if biophi_KabKabRelaxed['Antibody'].nunique()!=6:
        print( "biophi_KabKabRelaxed does not have 6 antibodies")
if biophi_KabKabStrict['Antibody'].nunique()!=6:
        print( "biophi_KabKabStrict does not have 6 antibodies")
if waltz['antibody'].nunique()!=6:
        print( "waltz does not have 6 antibodies")

biophi_KabKabRelaxed does not have 6 antibodies
biophi_KabKabStrict does not have 6 antibodies


In [9]:

biophi_KabKabRelaxed

,Antibody,Threshold,OASis Percentile,OASis Identity,Germline Content,Heavy V Germline,Heavy J Germline,Heavy OASis Percentile,Heavy OASis Identity,Heavy Non-human peptides,Heavy Germline Content
0,2Rs15d,relaxed,0.004000,0.392523,0.660870,IGHV3-23*01,IGHJ4*01,0.026766,0.392523,"(65,)",0.660870
1,Caplacizumab,relaxed,0.091667,0.608333,0.703125,IGHV3-23*04,IGHJ4*01,0.111833,0.608333,"(47,)",0.703125
2,Gontivimab_ALX-0171_C1,relaxed,0.012000,0.457627,0.603175,IGHV3-23*04,IGHJ6*01,0.056153,0.457627,"(64,)",0.603175
3,Gontivimab_ALX-0171_C2,relaxed,0.012000,0.457627,0.611111,IGHV3-23*04,IGHJ6*01,0.056153,0.457627,"(64,)",0.611111
4,Ozoralizumab_C1,relaxed,0.515103,0.794393,0.843478,IGHV3-74*01,IGHJ1*01,0.544888,0.794393,"(22,)",0.843478
5,Ozoralizumab_C2,relaxed,0.629514,0.850467,0.826087,IGHV3-23*04,IGHJ1*01,0.704262,0.850467,"(16,)",0.826087
6,Tarperprumig_ALXN1820_C1,relaxed,0.085522,0.582609,0.739837,IGHV3-11*01,IGHJ4*01,0.097783,0.582609,"(48,)",0.739837
7,Tarperprumig_ALXN1820_C2,relaxed,0.090721,0.603604,0.756303,IGHV3-23*01,IGHJ4*01,0.108523,0.603604,"(44,)",0.756303
8,Vobarilizumab_C1,relaxed,0.127354,0.654867,0.760331,IGHV3-66*01,IGHJ4*01,0.180761,0.654867,"(39,)",0.760331
9,Vobarilizumab_C2,relaxed,0.629514,0.850467,0.826087,IGHV3-23*04,IGHJ1*01,0.704262,0.850467,"(16,)",0.826087
